# Grupo 7 - Examen Práctico RL

Integrantes: 

- Diego Valenzuela 22309
- Daniel Dubón 22233
- Joaquín Puente 22296
- Christian Echeverria XXXX
- Pendiente XXXX

In [1]:
import numpy as np
from collections import defaultdict

# ESPECIFICACIÓN DEL MDP
# Estado: (nivel_inventario, dias_hasta_vencimiento, demanda_promedio_7dias)
# nivel_inventario:       [0, 10, 20, ..., 100]              — 10 niveles
# dias_hasta_vencimiento: [1, 7, 14, 30, 60]                 — 5 niveles
# demanda_promedio_7dias: [bajo, medio, alto, crítico]        — 4 niveles
# Total: 200 estados | Acciones: [0, 10, 20, 30, 40, 50] unidades — 6 acciones

def transition(state, action):
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 5, 'medio': 15, 'alto': 25, 'crítico': 40}
    daily_demand = demand_map[demand_level]
    new_inventory = min(100, max(0, inventory + action - daily_demand))
    new_days = max(1, days_to_expiry - 1)
    new_demand = demand_level
    return (new_inventory, new_days, new_demand)

def reward(state, action, next_state):
    new_inventory, new_days, _ = next_state
    inventory_reward = new_inventory * 0.5
    expiry_penalty = -10 if new_days <= 7 else 0
    order_penalty = -2 if action > 0 else 0
    return inventory_reward + expiry_penalty + order_penalty

def train(env, episodes=1000):
    Q = defaultdict(lambda: np.zeros(6))
    alpha = 0.9
    gamma = 0.99
    epsilon = 0.05
    for episode in range(episodes):
        state = env.reset()
        done = False
        while not done:
            if np.random.random() < epsilon:
                action = np.random.randint(6)
            else:
                action = np.argmax(Q[state])
            next_state, reward, done, _ = env.step(action)
            next_action = np.argmax(Q[next_state])
            td_target = reward + gamma * Q[next_state][next_action]
            Q[state][action] += alpha * (td_target - Q[state][action])
            state = next_state
    return Q

def preprocess_state(state):
    inventory, days_to_expiry, demand_level = state
    demand_map = {'bajo': 0, 'medio': 1, 'alto': 2, 'crítico': 3}
    features = np.array([
        inventory,
        days_to_expiry,
        demand_map[demand_level]
    ])
    return features

class LinearApproximator:
    def __init__(self, n_features=3, n_actions=6):
        self.weights = np.zeros((n_actions, n_features))
    def predict(self, features, action):
        return np.dot(self.weights[action], features)
    def update(self, features, action, target, alpha=0.01):
        prediction = self.predict(features, action)
        error = target - prediction
        self.weights[action] += alpha * error * features

def evaluate_policy(Q, env, episodes=100):
    total_rewards = []
    for episode in range(episodes):
        state = env.reset()
        episode_reward = 0
        done = False
        while not done:
            action = np.argmax(Q[state])
            next_state, reward, done, _ = env.step(action)
            episode_reward += reward
            state = next_state
        total_rewards.append(episode_reward)
    return {
        'mean_reward': np.mean(total_rewards),
        'std_reward': np.std(total_rewards),
        'min_reward': np.min(total_rewards)
    }

# MÉTRICAS DE ENTRENAMIENTO
# Episodio    Recompensa    Error TD    Política dominante
# 100         42.3          8.42        pedir_50
# 500         48.1          7.12        pedir_50
# 1000        51.7          5.21        pedir_50
# Varianza entre episodios: 0.8
# Política greedy: pedir 50 en 94% de estados

# RESULTADOS EN PRODUCCIÓN
resultados_produccion = {
    'stockouts_por_semana': 23,
    'productos_vencidos_por_semana': 41,
    'costo_almacenamiento_semanal': 8400,
    'costo_objetivo_semanal': 3200,
    'satisfaccion_cliente': 0.61
}

# RESULTADOS EN SIMULACIÓN
resultados_simulacion = {
    'mean_reward': 51.2,
    'std_reward': 0.9,
    'min_reward': 48.3
}

### Entregable 7.1 — Validación del gap simulación–producción

Se valida la dirección del análisis previo, pero se precisan las atribuciones y se separa lo demostrado por el código de lo que solo es una hipótesis. El cociente real de costo es $8400/3200 = 2.625$, es decir, aproximadamente **2.6×** el objetivo.


| Síntoma observado | Causa probable y componente | Evidencia en el código |
|---|---|---|
| **Costo de almacenamiento 2.6×** y política `pedir_50` en 94% | **Función de recompensa mal alineada.** Premia mantener inventario y casi no diferencia el tamaño del pedido. | `inventory_reward = new_inventory * 0.5` entrega hasta +50 por paso; `order_penalty = -2 if action > 0 else 0` cobra lo mismo por 10 que por 50 unidades. Por ello sobreabastecer puede maximizar el retorno simulado aunque sea caro en producción. |
| **41 productos vencidos/semana** | **Reward y modelo de transición incompletos.** El costo de vencer es binario y no proporcional a unidades; el estado tampoco representa lotes con edades distintas. | `expiry_penalty = -10 if new_days <= 7 else 0` aplica la misma penalización con una o cien unidades. `new_days = max(1, days_to_expiry - 1)` solo baja un contador y nunca descarta inventario vencido ni reinicia la vida útil de las unidades nuevas. El simulador no puede reproducir adecuadamente las 41 mermas. |
| **23 stockouts/semana** | **Reward sin costo de demanda perdida**, **transición que oculta el faltante** y **tabla Q sin generalización**. | `max(0, inventory + action - daily_demand)` recorta el inventario en cero, pero no conserva `lost_sales` ni penaliza su magnitud. Además, `defaultdict(lambda: np.zeros(6))` deja sin conocimiento los estados no vistos; la política greedy elige allí la acción 0. |
| **Satisfacción 0.61** | **Objetivo de entrenamiento/evaluación distinto al KPI real.** Servicio, stockouts y satisfacción no aparecen en el retorno. | `reward(...)` solo incluye inventario, umbral de vencimiento y un costo fijo por ordenar. `evaluate_policy(...)` reporta exclusivamente el retorno de ese mismo reward. |
| **51.2 estable en simulación pero mal resultado real** | **Evaluación sin cambio de distribución.** Se evalúa con el mismo `env` y las mismas simplificaciones usadas al entrenar. | `evaluate_policy(Q, env)` llama `env.reset()`/`env.step()` igual que `train`. La baja desviación (`0.9`) mide consistencia dentro de ese simulador, no robustez ante demanda o estados de producción. `new_demand = demand_level` además impide cambios de demanda dentro de cada episodio. |

**Precisión importante:** `epsilon = 0.05` solo explora **acciones**. No puede corregir por sí sola una transición que mantiene fija la demanda ni una distribución de `reset()` que omita estados. Aunque existe `LinearApproximator`, `train` y `evaluate_policy` usan exclusivamente la tabla `Q`, por lo que no hay generalización a tuplas nuevas.


### Entregable 7.2 — Política greedy en estados no visitados


In [2]:
# Auditoría reproducible de la estimación y de la política greedy.
TOTAL_ESTADOS_DECLARADOS = 200
PORCENTAJE_PEDIR_50 = 0.94
ACCIONES_EN_UNIDADES = [0, 10, 20, 30, 40, 50]

# Lectura prevista del dato agregado: si el 94% se calculó sobre los 200,
# 6% (= 12 estados) no elige pedir 50. Todo estado no visitado cae en este
# grupo porque conserva seis ceros y argmax desempata hacia el índice 0.
estados_que_no_eligen_50 = round(
    TOTAL_ESTADOS_DECLARADOS * (1 - PORCENTAJE_PEDIR_50)
)

q_estado_no_visitado = np.zeros(6)
accion_idx = int(np.argmax(q_estado_no_visitado))
accion_unidades = ACCIONES_EN_UNIDADES[accion_idx]
siguiente_estado_critico = transition((0, 1, 'crítico'), accion_unidades)

print(f"Estados fuera del 94%: {estados_que_no_eligen_50}/200")
print("Estimación probable de no visitados: ≈12/200 (6%), no un conteo exacto")
print(f"Q(s) no visitado: {q_estado_no_visitado}")
print(f"argmax -> índice {accion_idx} -> pedir {accion_unidades} unidades")
print(f"Desde (0, 1, 'crítico') se pasa a {siguiente_estado_critico}")


Estados fuera del 94%: 12/200
Estimación probable de no visitados: ≈12/200 (6%), no un conteo exacto
Q(s) no visitado: [0. 0. 0. 0. 0. 0.]
argmax -> índice 0 -> pedir 0 unidades
Desde (0, 1, 'crítico') se pasa a (0, 1, 'crítico')


#### 1. ¿Cuántos estados probablemente no se visitaron?

La estimación más directa que permite el dato agregado es **aproximadamente 12 de 200 estados (6%)**:

- La política reportada pide 50 en 94% de los 200 estados: $0.94(200)=188$.
- Quedan $200-188=12$ estados donde la acción greedy no es 50.
- Un estado no visitado conserva `Q[s] = [0,0,0,0,0,0]` y, por tanto, elige la acción 0; necesariamente queda fuera del 94% que elige 50.

Esta cifra es una **estimación**, no una igualdad demostrable. Formalmente, si el 94% fue calculado sobre los 200 estados, el código permite concluir que hay **como máximo 12** no visitados; algunos de esos 12 también podrían ser estados visitados que aprendieron legítimamente otra acción. La aproximación “12 no visitados” supone que las excepciones a la política dominante se deben principalmente a valores Q que nunca se actualizaron.

El riesgo de cobertura es plausible porque `new_demand = demand_level` impide cambiar de categoría de demanda dentro de un episodio, y `epsilon = 0.05` solo explora acciones, no categorías de estado. Sin embargo, el archivo no incluye `env.reset()`, el horizonte, la tabla Q final ni un conjunto `visited_states`; por eso no es válido fabricar un conteo exacto a partir de `len(Q)`. `Q` también puede contener estados consultados para *bootstrap* o evaluación, y la transición genera coordenadas fuera de la grilla, como días 29, 28, etc.

#### 2. ¿Qué hace la política greedy en un estado no visitado?

`Q = defaultdict(lambda: np.zeros(6))` crea seis ceros al consultar una tupla desconocida. Tanto `train` como `evaluate_policy` usan `np.argmax(Q[state])`. Cuando todos los valores empatan, NumPy devuelve la **primera** posición: índice 0. Con el orden declarado `[0, 10, 20, 30, 40, 50]`, el agente decide **pedir 0 unidades**. Crear la entrada al consultarla no significa que el estado haya sido aprendido.

#### 3. ¿Cómo genera stockouts?

En un estado de inventario bajo y demanda alta o crítica, ordenar cero hace que la demanda supere a `inventory + action`. La transición oculta el faltante al recortar con `max(0, ...)`: por ejemplo, desde `(0, 1, 'crítico')`, cuya demanda diaria es 40, la acción 0 devuelve de nuevo inventario 0. Como el reward no registra unidades de demanda insatisfecha y el código de producción no muestra aprendizaje en línea, la política puede repetir “no pedir” y prolongar el quiebre. Este mecanismo es consistente con los 23 stockouts observados, aunque el código no permite afirmar que explique exactamente los 23.

#### Controles de consistencia

1. El 94% debe tener como denominador los **200 estados** para sostener la estimación de 12. Si fue calculado solo sobre claves visitadas/presentes en `Q`, no informa cuántos estados faltan y el conteo exacto queda indeterminado.
2. La especificación tiene una contradicción adicional: `[0, 10, ..., 100]` contiene 11 valores, no 10; la grilla literal tendría $11 × 5 × 4 = 220$ estados. Aquí se usa 200 porque es el total exigido por el enunciado.
3. Para obtener el número real habría que registrar `visited_states.add(state)` dentro de `train` y comparar ese conjunto con la grilla válida; consultar después un `defaultdict` no es una medición confiable.

**Conclusión:** bajo la lectura prevista de la métrica del 94%, **unos 12 de 200 estados probablemente no se visitaron**. En cualquiera de ellos la política greedy elige **no pedir**, lo que convierte una falta de cobertura del entrenamiento en stockout cuando el estado desconocido tiene poco inventario y demanda elevada.


### Entregable 7.3 — Protocolo de evaluación mejorado

El protocolo actual (`evaluate_policy`) falla porque reutiliza el mismo `env` y la misma distribución de estados que `train`: mide consistencia interna del simulador, no transferencia a producción. Proponemos un protocolo de tres capas que se ejecuta **antes** de cualquier despliegue:

**Capa 1 — Métricas de cobertura de entrenamiento (gate previo a evaluar)**
- `estados_visitados / 200` y `pares_(s,a)_visitados / 1200`.
- Condición de aceptación: ≥ 95% de cobertura de estados y ≥ 90% de cobertura de pares `(s,a)`, con cobertura mínima del 80% dentro de cada categoría de demanda (incluyendo `crítico`), no solo en promedio global.
- Si no se cumple, el entrenamiento se rechaza antes de tocar métricas de recompensa — cobertura insuficiente invalida cualquier resultado posterior (ver Entregable 7.2).

**Capa 2 — Evaluación fuera de distribución (estrés dirigido)**
- Conjunto de prueba separado que sobre-muestrea deliberadamente los estados de baja frecuencia: inventario bajo (0–20) combinado con demanda `alto`/`crítico`, y `dias_hasta_vencimiento` bajo (1–7).
- Métrica: tasa de `stockout` simulado (`next_inventory == 0` bajo demanda no satisfecha) y tasa de vencimiento simulado, medidas por separado del reward agregado — un reward promedio saludable puede ocultar colas malas.
- Condición de aceptación: tasa de stockout simulado en el subconjunto de estrés no debe superar en más de 2× la tasa observada en el conjunto de prueba general.

**Capa 3 — Réplica de KPIs de negocio, no solo de reward**
- Traducir el reward a las mismas unidades que reporta producción: stockouts/semana, vencidos/semana, costo de almacenamiento/semana, satisfacción. `evaluate_policy` actual no calcula ninguno de estos directamente.
- Condición de aceptación: proyección de costo de almacenamiento simulado dentro de ±20% del costo objetivo (3200), antes de autorizar el paso a producción.
- Piloto controlado: desplegar la política nueva en un subconjunto pequeño de tiendas/productos durante 1–2 semanas y comparar KPIs reales contra la proyección de la Capa 3 antes del rollout completo.

Este protocolo detecta el gap actual (51.2 en simulación vs. 23 stockouts reales) porque la Capa 1 habría bloqueado el despliegue por cobertura insuficiente, y la Capa 3 habría expuesto que el reward optimizado no corresponde a los KPIs reales de negocio.


### Entregable 7.4 — Dictamen técnico para gerencia

**Para:** Gerencia de Operaciones — Cadena de Farmacias
**De:** Equipo de Consultoría Técnica, Grupo 7 (Evaluación en Producción)
**Asunto:** Causas del gap entre simulación y producción del agente de reabastecimiento

**Resumen ejecutivo.** El agente reporta una recompensa promedio de 51.2 en simulación, pero en producción genera 23 stockouts semanales, 41 unidades vencidas semanales y un costo de almacenamiento de \$8,400 (2.6× el objetivo de \$3,200). El gap no tiene una causa única: es la combinación de una función de recompensa mal alineada con el negocio, cobertura de entrenamiento insuficiente, y un protocolo de evaluación que no puede detectar ninguna de las dos anteriores porque reutiliza el mismo simulador y la misma distribución de estados usados en entrenamiento.

**Causas probables identificadas (evidencia en código, Entregable 7.1):**
1. **Recompensa desalineada** — `inventory_reward = new_inventory * 0.5` premia mantener inventario alto y `order_penalty` no escala con el tamaño del pedido; el agente converge a pedir siempre el máximo (94% de estados), lo que explica el sobrecosto de almacenamiento.
2. **Cobertura de entrenamiento insuficiente** — la política pide 0 unidades en el ~6% de estados restante (estimado en Entregable 7.2), consistente con estados nunca visitados donde `Q[s]` permanece en ceros; si esos estados corresponden a inventario bajo y demanda crítica, la acción por defecto (pedir 0) produce stockouts directos.
3. **Evaluación sin poder de detección** — `evaluate_policy` mide el mismo entorno y reward que `train`; no puede exponer el gap antes del despliegue porque no hay cambio de distribución ni traducción a KPIs de negocio (Entregable 7.3).

**Información que necesitamos confirmar de otros grupos para el diagnóstico definitivo:**
- **Grupo 3 (algoritmo):** ¿el error de implementación en `train` sesga la convergencia hacia una política subóptima adicional a la del reward, o es principalmente un problema de estabilidad numérica por `alpha = 0.9`? Necesitamos sus curvas originales vs. corregidas para separar el efecto del algoritmo del efecto del reward.
- **Grupo 5 (exploración):** número real de estados y pares `(s,a)` visitados en 1000 episodios con `epsilon = 0.05`. Esto confirma o refuta nuestra estimación de ~12/200 estados no visitados (Entregable 7.2) con datos directos en lugar de inferencia desde el 94%.
- **Grupo 2 (recompensa):** magnitud de cada componente de su función corregida, para estimar cuánto del sobrecosto de almacenamiento (2.6×) es atribuible al reward vs. a cobertura.
- **Grupo 1 (MDP):** si el estado corregido captura variabilidad de demanda, ¿cambia el mecanismo por el cual llegamos a estados de demanda crítica sin cobertura, o el problema de cobertura persiste bajo el nuevo espacio de estados?
- **Grupo 6 (convergencia):** su diagnóstico sobre si las curvas actuales muestran reward hacking puro o también estancamiento del algoritmo — determina si priorizar la corrección de recompensa o de algoritmo primero.

Sin esta información no podemos asignar pesos relativos a cada causa (Pregunta 2 de integración) ni proponer el plan mínimo viable con expectativas de mejora cuantificadas (Pregunta 3).


## Insumos pendientes de otros grupos

> Llenar esta tabla durante la sesión presencial. No borrar las preguntas — son la guía de qué pedirle a cada grupo.

| Grupo | Qué necesitamos | Pregunta concreta a hacerles | Estado |
|---|---|---|---|
| 1 (MDP) | Entregable 1.3 (MDP corregido) y 1.4 (tabla de tamaño de espacio) | ¿Cuántos estados tiene el MDP corregido? ¿La nueva transición de demanda hace que los estados críticos se visiten más seguido de forma natural, o el problema de cobertura persiste igual? | ⬜ Pendiente |
| 2 (Recompensa) | Entregable 2.3 (función corregida, 3+ componentes) y 2.4 (tabla comparativa de recompensa acumulada) | ¿Cuál es la magnitud de cada componente? ¿Cuánto reduce la recompensa acumulada de "pedir siempre 50" frente a una política razonable? | ⬜ Pendiente |
| 3 (Algoritmo) | Entregable 3.3 (train corregido) y 3.4 (curvas original vs. corregido) | ¿Cuál era el error exacto en `train`? ¿Qué algoritmo implementaba realmente el código original? ¿Cómo cambia el error TD y la varianza con la corrección? | ⬜ Pendiente |
| 4 (Aproximación) | Entregable 4.3 (preprocess_state mejorado) y 4.4 (tabla de MSE) | ¿Qué normalización usaron? ¿Sus características nuevas dependen de la representación de estado actual o de la corregida por el Grupo 1? | ⬜ Pendiente |
| 5 (Exploración) | Entregable 5.3 (métricas de cobertura) y 5.4 (stockouts atribuibles a no-cobertura) | ¿Cuántos estados y pares (s,a) de los 200/1200 se visitaron realmente en 1000 episodios con ε=0.05? ¿Cuántos de los 23 stockouts atribuyen a estados no visitados? | ⬜ Pendiente |
| 6 (Convergencia) | Entregable 6.1 (diagnóstico: óptimo local / reward hacking) y 6.4 (proyección de curvas post-corrección) | ¿Su diagnóstico es reward hacking, óptimo local, o ambos? ¿Cómo proyectan que cambie el error TD y la política dominante si se aplican las correcciones de los Grupos 1, 2 y 3 juntas? | ⬜ Pendiente |

**Regla de registro:** al recibir un insumo, pegarlo textualmente en la celda de la Pregunta de integración correspondiente (no resumir de memoria) y marcar el estado como ✅ aquí.


## Preguntas de integración

### Pregunta 1 — Diagnóstico sistémico

**Insumos usados:** MDP corregido (Grupo 1) · función de recompensa corregida (Grupo 2) · proyección de curvas (Grupo 6).

> **[PENDIENTE — completar en sesión presencial]**
> Pegar aquí, textualmente:
> - Entregable 1.3/1.4 del Grupo 1 (nueva representación de estado, tamaño de espacio).
> - Entregable 2.3/2.4 del Grupo 2 (componentes de reward, tabla comparativa de recompensa acumulada).
> - Entregable 6.4 del Grupo 6 (proyección de curvas).
>
> Luego responder: si se corrigen únicamente el MDP y el reward, sin tocar el algoritmo (Grupo 3, error crítico en `train`) ni la exploración (Grupo 5, cobertura de estados), ¿el agente aprende una política mejor? Nuestra hipótesis de trabajo (Entregables 7.1/7.2): **probablemente mejora el comportamiento de sobreabastecimiento (menos "pedir siempre 50"), pero no resuelve los stockouts**, porque estos últimos dependen de cobertura de exploración y de la corrección del algoritmo, no solo de qué tan bien esté definido el MDP o el reward. Confirmar o refutar esta hipótesis con las cifras concretas de los Grupos 1, 2 y 6 una vez recibidas.

### Pregunta 2 — Causa raíz

**Insumos usados:** algoritmo corregido (Grupo 3) · métricas de cobertura (Grupo 5) · análisis de gap (Grupo 7, propio: Entregables 7.1/7.2).

> **[PENDIENTE — completar en sesión presencial]**
> Pegar aquí, textualmente:
> - Entregable 3.3/3.4 del Grupo 3 (error identificado, curvas antes/después).
> - Entregable 5.3/5.4 del Grupo 5 (número de estados/pares (s,a) visitados, stockouts atribuidos a no-cobertura).
>
> Cruzar contra nuestra estimación propia (Entregable 7.2: ~12/200 estados sin visitar, política greedy "pedir 0" en esos estados) para decidir con evidencia cuantitativa de los tres grupos si el problema raíz es el error de implementación del algoritmo, la cobertura insuficiente, o el reward hacking (Entregable 7.1) — y en qué proporción cada uno.

### Pregunta 3 — Plan de corrección mínimo viable

**Insumos usados:** resultados de todos los grupos.

> **[PENDIENTE — completar en sesión presencial]**
> Estructura a llenar por cada corrección (agregar una fila por grupo consultado):
>
> | Prioridad | Componente que modifica | Grupo fuente | Métrica que mejora | Mejora esperada | Justificación |
> |---|---|---|---|---|---|
> | 1 | — | — | — | — | — |
> | 2 | — | — | — | — | — |
> | 3 | — | — | — | — | — |
>
> El plan debe ser ejecutable en dos semanas. Basar el orden de prioridad en el peso relativo que arroje la Pregunta 2.

### Pregunta 4 — Compatibilidad de correcciones

**Insumos usados:** MDP corregido (Grupo 1) · preprocesamiento mejorado (Grupo 4) · algoritmo corregido (Grupo 3).

> **[PENDIENTE — completar en sesión presencial]**
> Pegar aquí, textualmente:
> - Entregable 1.3 del Grupo 1 (nueva representación de estado).
> - Entregable 4.3 del Grupo 4 (preprocess_state mejorado, características nuevas).
> - Entregable 3.3 del Grupo 3 (train corregido).
>
> Punto de riesgo a verificar activamente: si el Grupo 1 cambia la tupla de estado (por ejemplo, agrega variabilidad de demanda o historial), el vector de características del Grupo 4 (`preprocess_state`) y el `n_features` de `LinearApproximator` deben actualizarse en consecuencia — de lo contrario el preprocesamiento del Grupo 4 quedará escrito para un espacio de estados que ya no existe. Confirmar con ambos grupos si coordinaron esto o si hay que resolver la incompatibilidad aquí.


## Reflexión grupal

> **[PENDIENTE — completar después de la sesión presencial]**
> Media página respondiendo: ¿qué cambió en nuestro diagnóstico inicial (Entregables 7.1/7.2, basado solo en el código y los agregados de producción) después de ver los resultados concretos de los otros grupos? Señalar específicamente qué hipótesis se confirmó, cuál se descartó o se matizó, y con el resultado de qué grupo.
